In [4]:
import os
import pandas as pd
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.oxml.ns import qn

# ================= 路径配置 =================
INPUT_IMG_DIR = os.path.join('..', 'output')
DATA_DIR = os.path.join('..', 'data_clean')
OUTPUT_DOC_DIR = os.path.join('..', 'doc')

os.makedirs(OUTPUT_DOC_DIR, exist_ok=True)
REPORT_PATH = os.path.join(OUTPUT_DOC_DIR, '宏观周期下的城市底色：35城财政与地产依赖度深度演变分析.docx')
CSV_PATH = os.path.join(DATA_DIR, 'engineered_panel_data.csv')

# ================= 初始化文档与字体 =================
doc = Document()

# 全局样式设置（修复中文方块乱码）
style = doc.styles['Normal']
style.font.name = 'Microsoft YaHei'
style._element.rPr.rFonts.set(qn('w:eastAsia'), 'Microsoft YaHei')

def apply_chinese_font(run):
    """强制为单个 Run 应用中文字体"""
    run.font.name = 'Microsoft YaHei'
    run._element.rPr.rFonts.set(qn('w:eastAsia'), 'Microsoft YaHei')

def add_section(heading, text):
    """添加标题与正文"""
    h = doc.add_heading(level=1)
    run_h = h.add_run(heading)
    apply_chinese_font(run_h)
    
    for paragraph in text.split('\n'):
        if paragraph.strip():
            if paragraph.startswith('- '):
                p = doc.add_paragraph(style='List Bullet')
                run_p = p.add_run(paragraph[2:])
            else:
                p = doc.add_paragraph()
                run_p = p.add_run(paragraph)
                p.paragraph_format.first_line_indent = Inches(0.25)
            apply_chinese_font(run_p)

def add_image(img_filename, width=6.0):
    """添加图片"""
    img_path = os.path.join(INPUT_IMG_DIR, img_filename)
    if os.path.exists(img_path):
        doc.add_picture(img_path, width=Inches(width))
        doc.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    else:
        p = doc.add_paragraph(f"[提示：未找到图片文件 {img_filename}]")
        p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        p.runs[0].font.color.rgb = RGBColor(255, 0, 0)
        apply_chinese_font(p.runs[0])

# ================= 报告内容组装 =================

# 标题
title = doc.add_heading(level=0)
run_title = title.add_run('宏观周期下的城市底色：35个主要城市财政与房地产依赖度深度演变分析')
apply_chinese_font(run_title)
title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

# 一、 引言与宏观财政底色
add_section('一、 引言与宏观财政底色：三大城市群的模式分野', """在探讨房地产依赖之前，必须先看清地方财政的“底层性格”。近5年（2020-2024）三大城市群的财政收支平均增长率对比，揭示了截然不同的区域治理模式：
- 珠三角的“克制与韧性”：财政支出平均增长率极低，呈现典型的“小政府、大市场”特征。这种“量入为出”的保守策略，使其在经济寒冬中保留了极强的抗风险能力。
- 长三角的“进击与扩张”：财政收入增长领跑，但支出增长更猛。这反映了长三角强烈的“政府引导”属性，通过积极的财政扩张强行孵化新产业。
- 京津冀的“紧平衡”：收支增速均在2%左右，处于稳健但承压的紧平衡状态。""")
add_image('clusters_growth_bar.png', width=5.5)

# 二、 财政缺口率时空演变深度剖析 (新增)
add_section('二、 财政缺口率时空演变深度剖析', """1. 时代浪潮下的极值城市扫描 (2006-2022)
通过提取关键宏观节点的财政缺口率（Gap to GDP）极值城市，我们能清晰看到区域经济重心的变迁。缺口率最小的城市常年被深圳、杭州等东部强市霸榜；而缺口率最大的城市则高度集中在西宁、兰州等西部与东北城市，反映了其对转移支付的结构性依赖。""")

# 动态生成极值表格
try:
    df = pd.read_csv(CSV_PATH)
    target_years = [2006, 2010, 2014, 2018, 2022]
    
    table = doc.add_table(rows=1, cols=5)
    table.style = 'Table Grid'
    hdr_cells = table.rows[0].cells
    headers = ['年份', '缺口率最大城市', '最大缺口率', '缺口率最小城市', '最小缺口率']
    for i, header in enumerate(headers):
        run = hdr_cells[i].paragraphs[0].add_run(header)
        run.font.bold = True
        apply_chinese_font(run)
        
    for y in target_years:
        if y in df['year'].values:
            df_y = df[df['year'] == y]
            max_row = df_y.loc[df_y['gap_to_gdp'].idxmax()]
            min_row = df_y.loc[df_y['gap_to_gdp'].idxmin()]
            
            row_cells = table.add_row().cells
            row_data = [
                str(y), 
                max_row['city'], f"{max_row['gap_to_gdp']:.2%}", 
                min_row['city'], f"{min_row['gap_to_gdp']:.2%}"
            ]
            for i, text in enumerate(row_data):
                run = row_cells[i].paragraphs[0].add_run(text)
                apply_chinese_font(run)
    doc.add_paragraph() # 增加空行
except Exception as e:
    p = doc.add_paragraph(f"[数据读取失败，无法生成表格: {e}]")
    apply_chinese_font(p.runs[0])

add_section('', """2. 顶流对决：北上广深一线城市缺口率演变
一线城市是全国经济的“压舱石”，但四者的财政底色走出了不同轨迹：
- 深圳的“弹性与自律”：缺口率长期处于最低位，展现出极强的自我修复与产业造血能力。
- 京沪的“稳健双子星”：走势高度趋同，缺口率常年稳定在极窄区间，得益于总部经济与金融中心的抗周期属性。
- 广州的“转型阵痛”：2014年后缺口率一路攀升，反映出其在传统商贸与制造业转型期，为维持城市能级进行了大量刚性支出。""")
add_image('tier1_gap_trend.png', width=6.0)

# 三、 典型城市切片
add_section('三、 典型城市切片：区域分化下的个体共振与背离', """1. 东部代表：克制与扩张的微观印证
广州完美印证了珠三角的“克制”，地产依赖度在深度调整期迅速下降，实现软着陆。而杭州与南京则印证了长三角的“扩张”，目前正处于“旧动能衰退、新动能接棒”的惊险跳跃期。""")
add_image('comparative_trend_东部_广州.png', width=5.0)
add_image('comparative_trend_东部_杭州.png', width=5.0)

add_section('', """2. 中部代表：强省会模式的周期阵痛
中部强省会模式过去高度依赖“土地-基建”循环。以郑州为例，地产依赖度曾飙升至35%以上。随着周期退潮，郑州与武汉的财政缺口率迅速拉大。长沙则相对稳健。""")
add_image('comparative_trend_中部_郑州.png', width=5.0)
add_image('comparative_trend_中部_武汉.png', width=5.0)

add_section('', """3. 西部代表：结构性双重挤压
西安呈现出典型的“双高”困境：财政缺口率常年高企，且地产依赖度曾高达30%。相比之下，成都的地产依赖度下降更为平滑，展现出西部领头羊的相对韧性。""")
add_image('comparative_trend_西部_西安.png', width=5.0)
add_image('comparative_trend_西部_成都.png', width=5.0)

# 四、 城市生态位聚类
add_section('四、 城市生态位聚类与深度空间分析', """基于2021-2024年的均值数据，我们将35城投射至十字象限图中：
- 第一象限：转型阵痛区（高依赖-高缺口）。集中在西部与部分中部城市。地产退潮后，财政缺口迅速恶化。
- 第二象限：内生乏力区（低依赖-高缺口）。以东北和部分老工业城市为主。人口流失导致地产引擎“被动熄火”。
- 第三象限：健康发展区（低依赖-低缺口）。被珠三角和部分东部一线包揽。强大的高端制造提供了充沛内生税源。
- 第四象限：传统红利区（高依赖-低缺口）。长三角核心城市及中西部强省会。目前仍能维持平衡，但必须加速“腾笼换鸟”。""")
add_image('quadrant_analysis_scatter.png', width=6.0)

# 五、 结论与建议
add_section('五、 结论与政策建议', """面对K型分化的空间格局，政策需因城施策：
- 针对高缺口区（第一、二象限）：短期内必须加大中央定向转移支付，坚决遏制隐性债务增量；中长期需通过跨区域产业转移重塑造血能力。
- 针对低缺口区（第三、四象限）：应赋予其更大的事权与财权自主性，抢抓当前财政尚有余力的窗口期，将沉淀在房地产中的资源坚决引导至新质生产力领域。""")

# 保存文档
doc.save(REPORT_PATH)
print(f"✅ 终极版 Word 报告已成功生成并保存至: {REPORT_PATH}")


✅ 终极版 Word 报告已成功生成并保存至: ..\doc\宏观周期下的城市底色：35城财政与地产依赖度深度演变分析.docx
